In [1]:
import pandas as pd
import numpy as np

In [2]:
# -- Carga de archivo --
df = pd.read_excel("NPE bruto 3.xlsx")

print("Filas iniciales:", len(df))
print("Columnas iniciales:", len(df.columns))

Filas iniciales: 1215
Columnas iniciales: 63


In [3]:
# -- Unificación de datos personales --

def combinar_columnas(df, col_1, col_2, nueva_col):
    df[nueva_col] = df[col_1].combine_first(df[col_2])

combinar_columnas(df, "Nombre", "Nombre2", "nombre_base")
combinar_columnas(df, "Apellidos", "Apellidos3", "apellidos_base")
combinar_columnas(df, "Correo electrónico", "Correo electrónico4", "correo_base")
combinar_columnas(df, "Teléfono", "Teléfono5", "telefono_base")
combinar_columnas(df, "Provincia", "Provincia6", "provincia_base")

df["nombre_completo"] = (
    df["nombre_base"].fillna("").astype(str).str.strip() + " " +
    df["apellidos_base"].fillna("").astype(str).str.strip()
).str.strip()

In [4]:
# -- Eliminación de nombres raros --

df = df[
    df["nombre_completo"].str.strip().str.match(
        r"^[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+( [A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*$",
        na=False
    )
]

df = df[
    ~df["nombre_completo"].str.lower().str.match(
        r"^([a-z])\1*$|^(test|asdf|xxx|null|none|na)$",
        na=False
    )
]

df = df[
    ~df["nombre_completo"].str.match(
        r"^[A-ZÁÉÍÓÚÜÑ] [A-ZÁÉÍÓÚÜÑ]$",
        na=False
    )
]

df["nombre_completo"] = df["nombre_completo"].str.title()

In [5]:
# -- Eliminación de personas sin contacto --
cols = ["Correo electrónico", "Teléfono", "Correo electrónico4", "Teléfono5"]
df = df.dropna(subset=cols, how="all")

print("Filas tras quitar sin contacto:", len(df))

Filas tras quitar sin contacto: 1192


In [7]:
# -- Eliminación de duplicados --
# contacto + nombre
cols = ["correo_base", "telefono_base", "nombre_completo"]
df = df.drop_duplicates()

print("Filas tras quitar duplicados:", len(df))

Filas tras quitar duplicados: 1192


In [8]:
# -- Eliminación de columnas de consentimiento, metadatos, info personal --
cols = [
    "Como eres menor de 18 años, se requerirá que tus padres o tutores completen el siguiente cuestionario por ti. \n\nEl COMITÉ PARALÍMPICO ESPAÑOL como Responsable del tratamiento le informa que la finalidad del tratamiento es la de gestionar la participación ",
    "Al completar este cuestionario, sus respuestas e información se compartirán con el personal específico del Comité Paralímpico Español.\n\nEl COMITÉ PARALÍMPICO ESPAÑOL como Responsable del tratamiento le informa que la finalidad del tratamiento es la de ges",
    "Por favor selecciona tu grupo de edad",
    "Dirección de residencia (ciudad y código postal)",
    "Dirección completa (ciudad/código postal)",
    "¿Cómo has conocido el cuestionario de deportistas paralímpicos?",
    "Other",
    "Nombre del Colegio donde has estudiado.",
    "Nombre de la asociación/club/entidad a la que estás vinculado/a.",
    "Perteneces a:",
    "Causa de la discapacidad",
    "Tengo discapacidad desde el año:",
    "completed",
    "Start Date (UTC)",
    "Submit Date (UTC)",
    "Network ID",
    "Tags",
    "Columna10"
]

df = df.drop(columns=cols)

In [9]:
# -- Corrección de fecha de nacimiento --
df["Fecha de nacimiento"] = pd.to_datetime(
    df["Fecha de nacimiento"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

In [10]:
# -- Codificaicón de discapacidades --

cols = [
    "Discapacidad visual",
    "Ataxia, atetosis y/o función muscular alterada.",
    "Otras discapacidades",
    "Función muscular alterada"
]
df[cols] = df[cols].notna().astype(int)

mask = (df[cols] == 1).any(axis=1)
df.loc[~mask, cols] = np.nan

In [11]:
# -- función para codificación --

def codificar_si_texto(serie, mapping, fillna_value=None):
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    serie = serie.map(mapping)
    if fillna_value is not None:
        serie = serie.fillna(fillna_value)
    return serie

In [12]:
# -- Codificación de discapacidad visual --
df["Mi discapacidad es:"] = codificar_si_texto(
    df["Mi discapacidad es:"],
    {
        "Resto visual conservado": 1,
        "Ceguera total": 2
    },
    fillna_value=0
)

In [13]:
# -- Codificación de Otras discapacidades --
nuevo_otro = (
    df[["Otras discapacidades", "Other7"]]
    .fillna("")
    .astype(str)
    .apply(lambda x: x.str.strip())
    .ne("")
    .any(axis=1)
    .astype(int)
)

# Eliminar columnas originales
df.drop(columns=["Otras discapacidades", "Other7"], inplace=True)

# Columna final limpia
df["otro"] = nuevo_otro

In [14]:
# -- Codificación de discapacidades de extremidades --
df["extremidad"] = df["Deficiencia en las extremidades afecta a:"].fillna("").str.strip().ne("").astype(int)

df.drop(columns=["Deficiencia en las extremidades afecta a:"], inplace=True)

In [15]:
# -- Codificación de interes deportivo --

deportes_catalogo = [
    "Atletismo", "Bádminton", "Baloncesto en silla de ruedas", "Boccia",
    "Ciclismo", "Escalada", "Esgrima en silla de ruedas",
    "Fútbol ciegos", "Goalball", "Powerlifting", "Hípica", "Judo",
    "Natación", "Piragüismo", "Remo", "Rugby en silla de ruedas",
    "Taekwondo", "Tenis de mesa", "Tenis en silla de ruedas",
    "Tiro con arco", "Tiro olímpico", "Triatlón", "Voleibol sentado",
    "Biatlón", "Curling en silla de ruedas", "Esquí alpino",
    "Esquí de fondo", "Hockey sobre hielo", "Snowboard"
]

cols_origen = ["El más interesante es:", "También me interesa:", "También me interesa:9"]

texto_deportes = (
    df[cols_origen]
    .fillna("")
    .astype(str)
    .agg(" | ".join, axis=1)
    .str.lower()
)

for deporte in deportes_catalogo:
    df[deporte] = texto_deportes.str.contains(deporte.lower(), regex=False).astype(int)

mask = (df[deportes_catalogo] == 1).any(axis=1)
df.loc[~mask, deportes_catalogo] = np.nan

In [16]:
# -- Codificación de práctica actual --
df["¿Actualmente participas o entrenas en algún deporte?"] = codificar_si_texto(
    df["¿Actualmente participas o entrenas en algún deporte?"],
    {
        "No": 0,
        "Sí, sólo entreno": 1,
        "Sí, solo entreno": 1,
        "Sí, entreno y compito": 2
    }
)

In [17]:
# -- Codificación de tiempo de deporte --
df["¿Cuánto tiempo a la semana dedicas al deporte?"] = codificar_si_texto(
    df["¿Cuánto tiempo a la semana dedicas al deporte?"],
    {
        "Menos de 3 veces por semana": 1,
        "3 veces por semana": 2,
        "Más de 3 veces por semana": 2,
        "3 o más veces por semana": 2
    }
)

In [18]:
# -- Codificación de Clasificación por panel --
df["¿Has sido clasificado por un panel de clasificación?"] = codificar_si_texto(
    df["¿Has sido clasificado por un panel de clasificación?"],
    {
        "No tengo clasificación": 0,
        "No estoy seguro": 1,
        "Sí he pasado clasificación nacional": 2,
        "Sí, he pasado clasificación nacional": 2,
        "Sí he pasado clasificación internacional": 3,
        "Sí, he pasado clasificación internacional": 3
    },
    fillna_value=0
)

In [19]:
# -- Renombrar columnas --
df.rename(columns={
    "nombre_completo": "nombre_completo",
    "Fecha de nacimiento": "fecha_nacimiento",
    "¿Actualmente perteneces o has pertenecido a las Fuerzas Armadas o a algún cuerpo de seguridad del estado?": "cuerpo_seguridad",
    "Describe tu discapacidad:": "descripcion_discapacidad",
    "Discapacidad visual": "discapacidad_visual",
    "Ataxia, atetosis y/o función muscular alterada.": "problema_coordinacion",
    "Otras discapacidades": "otro",
    "Mi discapacidad es:": "nivel_discapacidad_visual",
    "Función muscular alterada": "falta_fuerza",
    "Dinos cual.": "deporte_previo",
    "¿Actualmente participas o entrenas en algún deporte?": "practica_deporte",
    "¿Qué deporte?": "deporte_actual",
    "¿Cuánto tiempo a la semana dedicas al deporte?": "tiempo_deporte_semana",
    "¿Has competido alguna vez?": "ha_competido",
    "¿Has sido clasificado por un panel de clasificación?": "clasificado_panel",
    "¿Tienes actualmente un entrenador?": "entrenador_actual",
    "¿Te gusta competir?": "interes_competir_paralimpico",
    "Provincia6": "provincia",
    "Isla": "isla",
    "Atletismo": "atletismo",
    "Bádminton": "badminton",
    "Baloncesto en silla de ruedas": "baloncesto_silla_ruedas",
    "Boccia": "boccia",
    "Ciclismo": "ciclismo",
    "Escalada": "escalada",
    "Esgrima en silla de ruedas": "esgrima_silla_ruedas",
    "Fútbol ciegos": "futbol_ciegos",
    "Goalball": "goalball",
    "Powerlifting": "powerlifting",
    "Hípica": "hipica",
    "Judo": "judo",
    "Natación": "natacion",
    "Piragüismo": "piraguismo",
    "Remo": "remo",
    "Rugby en silla de ruedas": "rugby_silla_ruedas",
    "Taekwondo": "taekwondo",
    "Tenis de mesa": "tenis_mesa",
    "Tenis en silla de ruedas": "tenis_silla_ruedas",
    "Tiro con arco": "tiro_arco",
    "Tiro olímpico": "tiro_olimpico",
    "Triatlón": "triatlon",
    "Voleibol sentado": "voleibol_sentado",
    "Biatlón": "biatlon",
    "Curling en silla de ruedas": "curling_silla_ruedas",
    "Esquí alpino": "esqui_alpino",
    "Esquí de fondo": "esqui_fondo",
    "Hockey sobre hielo": "hockey_hielo",
    "Snowboard": "snowboard"
}, inplace=True)

In [20]:
# -- Eliminación de columnas --
cols = [
    "COMENTARIOS",
    "LLAMAR/CORREO",
    "ESTRELLAS",
    "Deporte",
    "Provincia",
    "¿Tienes nacionalidad española?",
    "Provincia6",
    "isla",
    "correo_base",
    "telefono_base",
    "provincia",
    "Nombre",
    "Apellidos",
    "Correo electrónico",
    "Teléfono",
    "Nombre2",
    "Apellidos3",
    "Correo electrónico4",
    "Teléfono5",
    "nombre_base",
    "apellidos_base",
    "provincia_base",
    "El más interesante es:",
    "También me interesa:",
    "También me interesa:9",
    "Por favor detalla tu alteración del equilibrio",
    "¿La discapacidad intelectual fue diagnosticada antes de los 18 años?",
    "nivel_discapacidad_visual",
    "La Deficiencia en las extremidades es",
    "La Deficiencia en las extremidades es8",
    "Deficiencia en las extremidades es",
    "deporte_previo",
    "¿Dónde entrenas?",
    "ha_competido",
    "¿Has participado en algún deporte previo a la discapacidad?",
    "El propósito del cuestionario es detectar un deporte paralímpico en el que puedas competir con las mayores garantías.\nUn deportista debe tener una de estas discapacidades para competir en deportes paralímpicos. Selecciona todas las respuestas válidas."
]
df.drop(columns=cols, axis=1, inplace=True, errors="ignore")

In [21]:
# -- Reordenamos columnas --
cols = df.columns.tolist()
cols.insert(0, cols.pop(cols.index("nombre_completo")))
df = df[cols]

cols = list(df.columns)

# quitar las columnas que queremos mover
cols.remove("otro")
cols.remove("extremidad")

# encontrar posición de referencia
idx = cols.index("falta_fuerza")

# insertar justo después
cols.insert(idx + 1, "otro")
cols.insert(idx + 2, "extremidad")

# reordenar dataframe
df = df[cols]

In [22]:
# -- Resultado Final y guardado --
df.to_excel("NPE_bruto_3_limpio.xlsx", index=False)

print("Archivo generado: NPE_bruto_3_limpio.xlsx")
print("Filas finales:", len(df))
print("Columnas finales:", len(df.columns))

Archivo generado: NPE_bruto_3_limpio.xlsx
Filas finales: 1192
Columnas finales: 44
